In [19]:
import json
import re
import time
from collections.abc import Callable, Mapping, Sequence
from typing import Literal

from pydantic import BaseModel, Field, ValidationError

### Task 1

In [20]:
class Critique(BaseModel):
    """Structured critique returned by a model."""

    passed: bool
    issues: list[str] = Field(default_factory=list)
    severity: int = Field(default=0, ge=0, le=3)


def _strip_fences(text: str) -> str:
    return re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.IGNORECASE)


def parse_critique(
    raw_output: str,
    retry_fn: Callable[[str], str] | None = None,
) -> Critique:
    """Parse raw model output into Critique, retrying once on invalid JSON."""
    try:
        return Critique.model_validate_json(_strip_fences(raw_output))
    except ValidationError as err:
        if retry_fn is None:
            raise
        retry_prompt = (
            f"Your previous response was not valid Critique JSON ({err}).\n"
            f"Return ONLY valid JSON matching: {Critique.model_json_schema()}\n\n"
            f"Invalid response:\n{raw_output}"
        )
        return Critique.model_validate_json(_strip_fences(retry_fn(retry_prompt)))

In [21]:
# 1. Valid JSON (no retry)
valid_json = '```json\n{"passed": true, "issues": [], "severity": 0}\n```'
assert parse_critique(valid_json).passed is True

# 2. Invalid JSON -> retries once and succeeds
attempts: list[str] = []


def mock_retry(prompt: str) -> str:
    attempts.append(prompt)
    return '{"passed": false, "issues": ["Missing edge case"], "severity": 2}'


critique = parse_critique("{bad json", retry_fn=mock_retry)
assert len(attempts) == 1
assert critique == Critique(passed=False, issues=["Missing edge case"], severity=2)

# 3. Still invalid after 1 retry -> raises ValidationError
try:
    parse_critique("{bad json", retry_fn=lambda _: "still bad")
except ValidationError:
    print("Task 1 checks passed!")
else:
    raise AssertionError("Expected ValidationError after retry failed")

Task 1 checks passed!


### Task 2

In [22]:
class UserProfile(BaseModel):
    """Structured output schema for deterministic verification."""

    username: str = Field(min_length=3)
    email: str
    age: int = Field(ge=13, le=120)
    role: Literal["admin", "editor", "viewer"]


def verify_user_profile(raw_output: str) -> list[str]:
    """Deterministically verify JSON output and return model-readable error strings."""
    try:
        profile = UserProfile.model_validate_json(_strip_fences(raw_output))
    except ValidationError as err:
        errors: list[str] = []
        for e in err.errors():
            field = ".".join(str(part) for part in e["loc"]) or "json"
            errors.append(f"Field '{field}': {e['msg']} (got {e.get('input')!r}).")
        return errors

    errors = []
    if "@" not in profile.email or "." not in profile.email.split("@")[-1]:
        errors.append(f"Field 'email': invalid email format (got {profile.email!r}).")
    if profile.role == "admin" and profile.age < 21:
        errors.append(
            f"Business rule 'role/age': 'admin' requires age >= 21 (got age={profile.age})."
        )
    return errors

In [23]:
# 1. Valid profile -> no errors
ok_profile = '{"username": "alice", "email": "alice@example.com", "age": 25, "role": "admin"}'
assert verify_user_profile(ok_profile) == []

# 2. Invalid JSON -> specific JSON error string
json_errors = verify_user_profile("{bad json")
assert len(json_errors) == 1 and "Field 'json':" in json_errors[0]

# 3. Schema + business rule errors -> specific field/rule messages
schema_errors = verify_user_profile(
    '{"username": "al", "email": "bad", "age": 19, "role": "owner"}'
)
assert any("Field 'username':" in e for e in schema_errors)
assert any("Field 'role':" in e for e in schema_errors)

rule_errors = verify_user_profile(
    '{"username": "alice", "email": "not-an-email", "age": 19, "role": "admin"}'
)
assert len(rule_errors) == 2
assert any("Field 'email':" in e for e in rule_errors)
assert any("Business rule 'role/age':" in e for e in rule_errors)
print("Task 2 checks passed!")

Task 2 checks passed!


### Task 3

In [24]:
class ReflectionResult(BaseModel):
    """Final output and status from the generate-critique-revise loop."""

    status: Literal["accepted", "unresolved"]
    output: str
    rounds: int
    critique: Critique


def reflect_loop(
    task: str,
    llm_fn: Callable[[str], str],
    max_rounds: int = 3,
) -> ReflectionResult:
    """Run generate -> critique -> revise up to max_rounds, returning accepted or unresolved."""
    if max_rounds < 1:
        raise ValueError("max_rounds must be >= 1")

    output = llm_fn(f"Task:\n{task}\n\nGenerate the best initial solution.")
    critique = Critique(passed=False)

    for round_num in range(1, max_rounds + 1):
        critique_prompt = (
            f"Task:\n{task}\n\nCurrent output:\n{output}\n\n"
            f"Critique it and return ONLY valid JSON matching: {Critique.model_json_schema()}"
        )
        critique = parse_critique(llm_fn(critique_prompt), retry_fn=llm_fn)
        if critique.passed:
            return ReflectionResult(
                status="accepted",
                output=output,
                rounds=round_num,
                critique=critique,
            )
        if round_num < max_rounds:
            revise_prompt = (
                f"Task:\n{task}\n\nCurrent output:\n{output}\n\n"
                f"Fix these issues (severity {critique.severity}): {critique.issues}"
            )
            output = llm_fn(revise_prompt)

    return ReflectionResult(
        status="unresolved",
        output=output,
        rounds=max_rounds,
        critique=critique,
    )

In [25]:
# 1. Accepted after 1 revision
accept_responses = iter(
    [
        "draft v1",
        '{"passed": false, "issues": ["Handle empty input"], "severity": 2}',
        "draft v2 (handles empty input)",
        '{"passed": true, "issues": [], "severity": 0}',
    ]
)
res_ok = reflect_loop("Write safe_avg", lambda _p: next(accept_responses), max_rounds=3)
assert res_ok.status == "accepted" and res_ok.rounds == 2
assert res_ok.output == "draft v2 (handles empty input)"

# 2. Hits round cap -> unresolved
fail_responses = iter(
    [
        "draft v1",
        '{"passed": false, "issues": ["Still buggy"], "severity": 3}',
        "draft v2",
        '{"passed": false, "issues": ["Still buggy"], "severity": 3}',
    ]
)
res_fail = reflect_loop("Write safe_avg", lambda _p: next(fail_responses), max_rounds=2)
assert res_fail.status == "unresolved" and res_fail.rounds == 2
assert res_fail.critique.passed is False
print("Task 3 checks passed!")

Task 3 checks passed!


### Task 4

In [26]:
def normalize_draft(draft: str) -> str:
    """Normalize draft text by lowercasing and collapsing whitespace."""
    return " ".join(draft.lower().split())


def detect_oscillation(drafts: Sequence[str]) -> bool:
    """Return True if any normalized draft repeats across rounds."""
    normalized = [normalize_draft(d) for d in drafts]
    return len(set(normalized)) < len(normalized)

In [27]:
assert detect_oscillation([]) is False
assert detect_oscillation(["only one"]) is False
assert detect_oscillation(["Draft 1", "Draft 2", "Draft 3"]) is False
assert detect_oscillation(["Draft A", "Draft B", "  draft   a \n"]) is True
assert detect_oscillation(["Same", "same"]) is True
print("Task 4 checks passed!")

Task 4 checks passed!


### Task 5

In [28]:
class ScoredDraft(BaseModel):
    """Highest-scoring draft selected across reflection rounds."""

    draft: str
    score: float
    round_num: int


def select_best_draft(
    drafts: Sequence[str],
    score_fn: Callable[[str], float],
) -> ScoredDraft:
    """Score each draft and return the highest-scoring version (best-so-far)."""
    if not drafts:
        raise ValueError("drafts must not be empty")

    best = ScoredDraft(draft=drafts[0], score=score_fn(drafts[0]), round_num=1)
    for round_num, draft in enumerate(drafts[1:], start=2):
        score = score_fn(draft)
        if score > best.score:
            best = ScoredDraft(draft=draft, score=score, round_num=round_num)
    return best

In [29]:
scores = {"v1": 0.5, "v2": 0.92, "v3": 0.70}
best = select_best_draft(["v1", "v2", "v3"], score_fn=lambda d: scores[d])
assert best == ScoredDraft(draft="v2", score=0.92, round_num=2)

tie_scores = {"a": 0.8, "b": 0.8}
assert select_best_draft(["a", "b"], score_fn=lambda d: tie_scores[d]).draft == "a"
print("Task 5 checks passed!")

Task 5 checks passed!


### Task 6

In [30]:
class Citation(BaseModel):
    """Cited source id and supporting quote."""

    source_id: str
    quote: str


def verify_citations(
    citations: Sequence[Citation],
    sources: Mapping[str, str],
) -> list[str]:
    """Check that every cited source_id exists and its quote appears in that source."""
    errors: list[str] = []
    for c in citations:
        if c.source_id not in sources:
            errors.append(f"Unknown source_id {c.source_id!r}.")
            continue
        if not c.quote.strip() or normalize_draft(c.quote) not in normalize_draft(
            sources[c.source_id]
        ):
            errors.append(f"Quote {c.quote!r} not found in source {c.source_id!r}.")
    return errors

In [31]:
sources = {"s1": "Revenue grew 20% in Q4 2025.", "s2": "Latency dropped to 12ms."}

assert verify_citations([Citation(source_id="s1", quote="grew 20% in Q4")], sources) == []

bad = verify_citations(
    [
        Citation(source_id="s9", quote="anything"),
        Citation(source_id="s2", quote="Latency rose to 50ms"),
        Citation(source_id="s1", quote=""),
    ],
    sources,
)
assert len(bad) == 3
assert "Unknown source_id 's9'" in bad[0]
assert "not found in source 's2'" in bad[1]
assert "not found in source 's1'" in bad[2]
print("Task 6 checks passed!")

Task 6 checks passed!


### Task 7

In [32]:
class CriterionScore(BaseModel):
    """Score for one rubric criterion backed by a quote from the draft."""

    criterion: str
    score: int = Field(ge=1, le=5)
    evidence_quote: str = Field(min_length=1)


class RubricVerdict(BaseModel):
    """Per-criterion scores returned by the rubric judge."""

    scores: list[CriterionScore]

    @property
    def average_score(self) -> float:
        return sum(s.score for s in self.scores) / len(self.scores) if self.scores else 0.0


def judge_rubric(
    draft: str,
    criteria: Sequence[str],
    llm_fn: Callable[[str], str],
) -> RubricVerdict:
    """Score a draft per criterion and verify each evidence quote appears in the draft."""
    prompt = (
        f"Score this draft (1-5) on criteria {list(criteria)} with a verbatim evidence_quote.\n"
        f"Return ONLY JSON matching: {RubricVerdict.model_json_schema()}\n\n"
        f"Draft:\n{draft}"
    )
    verdict = RubricVerdict.model_validate_json(_strip_fences(llm_fn(prompt)))
    missing = set(criteria) - {s.criterion for s in verdict.scores}
    if missing:
        raise ValueError(f"Missing rubric criteria: {sorted(missing)}")

    norm_draft = normalize_draft(draft)
    for item in verdict.scores:
        if normalize_draft(item.evidence_quote) not in norm_draft:
            raise ValueError(f"Evidence quote not found in draft: {item.evidence_quote!r}")
    return verdict

In [33]:
def seeded_judge_llm(prompt: str) -> str:
    draft = prompt.split("Draft:\n", 1)[1]
    has_div = "/ len(nums)" in draft
    edge_quote = (
        "if nums" if "if nums" in draft else ("if not nums" if "if not nums" in draft else "")
    )
    return json.dumps(
        {
            "scores": [
                {
                    "criterion": "correctness",
                    "score": 5 if has_div else 1,
                    "evidence_quote": "/ len(nums)" if has_div else "return sum(nums)",
                },
                {
                    "criterion": "edge_cases",
                    "score": 5 if edge_quote else 1,
                    "evidence_quote": edge_quote or draft.splitlines()[0],
                },
            ]
        }
    )


criteria = ["correctness", "edge_cases"]
flawed_drafts = [
    "def avg(nums): return sum(nums) / len(nums)",
    "def avg(nums): return sum(nums)",
]
good_drafts = [
    "def avg(nums): return sum(nums) / len(nums) if nums else 0.0",
    "def avg(nums):\n    if not nums:\n        return 0.0\n    return sum(nums) / len(nums)",
]

flawed_verdicts = [judge_rubric(d, criteria, seeded_judge_llm) for d in flawed_drafts]
good_verdicts = [judge_rubric(d, criteria, seeded_judge_llm) for d in good_drafts]

assert all(len(v.scores) == 2 for v in flawed_verdicts + good_verdicts)
assert max(v.average_score for v in flawed_verdicts) < min(v.average_score for v in good_verdicts)

bad_quote_json = (
    '{"scores": [{"criterion": "correctness", "score": 5, "evidence_quote": "not in draft"}]}'
)
try:
    judge_rubric("def avg(nums): return 0", ["correctness"], lambda _p: bad_quote_json)
except ValueError:
    print("Task 7 checks passed!")
else:
    raise AssertionError("Expected ValueError for ungrounded evidence quote")

Task 7 checks passed!


### Task 8

In [34]:
PairChoice = Literal["A", "B", "TIE"]


def compare_pairwise(
    candidate_a: str,
    candidate_b: str,
    judge_fn: Callable[[str, str], PairChoice],
) -> PairChoice | None:
    """Compare candidates in both orders; return verdict only when they agree, else None."""
    forward = judge_fn(candidate_a, candidate_b)
    reverse = judge_fn(candidate_b, candidate_a)
    flip: dict[PairChoice, PairChoice] = {"A": "B", "B": "A", "TIE": "TIE"}
    return forward if forward == flip[reverse] else None

In [35]:
def length_judge(first: str, second: str) -> PairChoice:
    if len(first) == len(second):
        return "TIE"
    return "A" if len(first) > len(second) else "B"


assert compare_pairwise("longer draft", "short", length_judge) == "A"
assert compare_pairwise("short", "longer draft", length_judge) == "B"
assert compare_pairwise("same", "size", length_judge) == "TIE"
assert compare_pairwise("opt1", "opt2", lambda _a, _b: "A") is None
print("Task 8 checks passed!")

Task 8 checks passed!


### Task 9

In [36]:
class LessonRecord(BaseModel):
    """Structured lesson record from a failed run, ready for episodic memory."""

    action_type: str = "reflection_lesson"
    task: str
    failure_reason: str
    lesson: str
    confidence: float = Field(ge=0.0, le=1.0)
    tags: list[str] = Field(default_factory=list)


def extract_lesson(
    task: str,
    failed_run: ReflectionResult,
    tags: Sequence[str] = (),
) -> LessonRecord:
    """Turn an unresolved ReflectionResult into a short, episodic-memory-ready lesson."""
    if failed_run.status != "unresolved":
        raise ValueError("Lesson extraction requires a failed ('unresolved') run")

    issues = failed_run.critique.issues
    reason = "; ".join(issues[:3]) if issues else "Unresolved after reaching round cap"
    lesson = f"Check and fix before finalizing: {reason}" if issues else "Review edge cases earlier"
    confidence = 0.85 if issues else 0.4

    return LessonRecord(
        task=task.strip(),
        failure_reason=reason,
        lesson=lesson,
        confidence=confidence,
        tags=list(tags),
    )

In [37]:
failed = ReflectionResult(
    status="unresolved",
    output="def avg(nums): return sum(nums) / len(nums)",
    rounds=2,
    critique=Critique(passed=False, issues=["ZeroDivisionError on empty list"], severity=3),
)
record = extract_lesson("Write safe_avg", failed, tags=["python", "edge_case"])
assert 0.0 <= record.confidence <= 1.0 and record.confidence == 0.85
assert "ZeroDivisionError" in record.lesson
assert record.model_dump()["tags"] == ["python", "edge_case"]

vague_failed = ReflectionResult(
    status="unresolved",
    output="bad",
    rounds=2,
    critique=Critique(passed=False, issues=[], severity=2),
)
assert extract_lesson("Write safe_avg", vague_failed).confidence == 0.4

try:
    extract_lesson(
        "ok",
        ReflectionResult(status="accepted", output="ok", rounds=1, critique=Critique(passed=True)),
    )
except ValueError:
    print("Task 9 checks passed!")
else:
    raise AssertionError("Expected ValueError for accepted run")

Task 9 checks passed!


### Task 10

In [38]:
class ABVariantSummary(BaseModel):
    """Aggregated A/B metrics for baseline vs reflection."""

    variant: str
    avg_quality: float
    avg_rounds: float
    avg_latency_ms: float
    total_cost_usd: float


def _track_calls(
    llm_fn: Callable[[str], str],
    cost_per_token: float,
) -> tuple[Callable[[str], str], list[float]]:
    costs: list[float] = []

    def wrapped(prompt: str) -> str:
        reply = llm_fn(prompt)
        costs.append(((len(prompt) + len(reply)) / 4) * cost_per_token)
        return reply

    return wrapped, costs


def run_ab_eval(
    tasks: Sequence[str],
    baseline_llm: Callable[[str], str],
    reflection_llm: Callable[[str], str],
    score_fn: Callable[[str, str], float],
    max_rounds: int = 3,
    cost_per_token: float = 1e-6,
) -> list[ABVariantSummary]:
    """Compare baseline vs reflection on quality, rounds, latency, and cost."""
    if not tasks:
        raise ValueError("tasks must not be empty")

    def eval_variant(
        name: str,
        runner: Callable[[str, Callable[[str], str]], tuple[str, int]],
        llm_fn: Callable[[str], str],
    ) -> ABVariantSummary:
        qualities, rounds_list, latencies_ms = [], [], []
        total_cost = 0.0
        for task in tasks:
            tracked, call_costs = _track_calls(llm_fn, cost_per_token)
            start = time.perf_counter()
            output, rounds = runner(task, tracked)
            latencies_ms.append((time.perf_counter() - start) * 1000)
            qualities.append(score_fn(task, output))
            rounds_list.append(rounds)
            total_cost += sum(call_costs)
        n = len(tasks)
        return ABVariantSummary(
            variant=name,
            avg_quality=sum(qualities) / n,
            avg_rounds=sum(rounds_list) / n,
            avg_latency_ms=sum(latencies_ms) / n,
            total_cost_usd=total_cost,
        )

    def run_reflection(task: str, fn: Callable[[str], str]) -> tuple[str, int]:
        res = reflect_loop(task, fn, max_rounds=max_rounds)
        return res.output, res.rounds

    summaries = [
        eval_variant("baseline", lambda t, fn: (fn(t), 1), baseline_llm),
        eval_variant("reflection", run_reflection, reflection_llm),
    ]
    header = (
        f"{'Variant':<12} | {'Quality':>7} | {'Rounds':>6} | {'Latency(ms)':>11} | {'Cost($)':>10}"
    )
    print(header)
    print("-" * len(header))
    for s in summaries:
        print(
            f"{s.variant:<12} | {s.avg_quality:>7.2f} | {s.avg_rounds:>6.2f} "
            f"| {s.avg_latency_ms:>11.2f} | {s.total_cost_usd:>10.6f}"
        )
    return summaries

In [39]:
def demo_reflection_llm(prompt: str) -> str:
    if "Critique it" in prompt:
        if "robust" in prompt:
            return '{"passed": true, "issues": [], "severity": 0}'
        return '{"passed": false, "issues": ["Handle edge cases"], "severity": 2}'
    if "Fix these issues" in prompt:
        return "robust solution"
    return "basic draft"


ab_tasks = ["Write safe_avg", "Write safe_div"]
ab_results = run_ab_eval(
    ab_tasks,
    baseline_llm=lambda _p: "basic draft",
    reflection_llm=demo_reflection_llm,
    score_fn=lambda _t, out: 1.0 if "robust" in out else 0.5,
)
baseline_sum, reflection_sum = ab_results
assert baseline_sum.avg_quality == 0.5 and baseline_sum.avg_rounds == 1.0
assert reflection_sum.avg_quality == 1.0 and reflection_sum.avg_rounds == 2.0
assert reflection_sum.total_cost_usd > baseline_sum.total_cost_usd
print("Task 10 checks passed!")

Variant      | Quality | Rounds | Latency(ms) |    Cost($)
----------------------------------------------------------
baseline     |    0.50 |   1.00 |        0.00 |   0.000012
reflection   |    1.00 |   2.00 |        1.82 |   0.000611
Task 10 checks passed!
